# PhysAR-IDS -- NB3b (v6): Attack 1 -- per-class confusion matrices, clean vs PGD (eps = 0.1)

Add-on to NB3. It **reuses the saved surrogates** (`surrogate_mlp_v6_seed*.pt`), regenerates the PGD eps=0.1 adversarial test set for each seed, refits the Optuna-tuned XGBoost on the same GPU backend, and reports where each true class goes under attack.
Question it answers: ransomware has 0.000 evasion to *benign* yet its recall falls to ~0.23 -- which classes absorb those rows?

Checks: the seed-42 adversarial set regenerated here must equal `X_adv_pgd_eps01_v6.npy`, and the tuned clean F1 must match NB3. Runs in a few minutes.

In [ ]:
import os, json, time, warnings, random
import numpy as np, pandas as pd, joblib
import torch, torch.nn as nn, torch.nn.functional as F
import xgboost as xgb
from sklearn.metrics import f1_score, confusion_matrix
import matplotlib; import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation'); os.chdir(PROJECT_DIR)
PIPELINE_VERSION = 'v6'; SEEDS = [42, 7, 123, 2024, 99]; EPS = 0.1; PGD_ITERS = 10
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with warnings.catch_warnings(record=True) as _c:
    warnings.simplefilter('always'); _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _c)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl'); assert data['PIPELINE_VERSION'] == PIPELINE_VERSION
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test']); N_FEAT = X_train.shape[1]
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)

class SurrogateMLP(nn.Module):
    def __init__(self, n_features, n_classes=6, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_features, hidden), nn.ReLU(), nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, n_classes))
    def forward(self, x): return self.net(x)
def pgd_attack(model, X, y, eps, n_iter=PGD_ITERS):
    model.eval(); alpha = eps / 4
    yt = torch.from_numpy(y).long().to(DEVICE); X0 = torch.from_numpy(X).float().to(DEVICE); Xa = X0.clone()
    for _ in range(n_iter):
        Xa.requires_grad_(True)
        g = torch.autograd.grad(F.cross_entropy(model(Xa), yt), Xa)[0]
        Xa = Xa.detach() + alpha * g.sign(); Xa = X0 + torch.clamp(Xa - X0, -eps, eps)
    return Xa.detach().cpu().numpy()
print(f'Train {X_train.shape} Test {X_test.shape} | backend {BACKEND} | device {DEVICE}')

## 1. Regenerate adversarial sets from the saved surrogates, refit the tuned target, collect predictions

In [ ]:
CM_CLEAN, CM_ADV, CHECK = {}, {}, {}
for seed in SEEDS:
    t0 = time.time()
    sur = SurrogateMLP(N_FEAT).to(DEVICE); sur.load_state_dict(torch.load(f'surrogate_mlp_{PIPELINE_VERSION}_seed{seed}.pt', map_location=DEVICE)); sur.eval()
    Xa = pgd_attack(sur, X_test, y_test, EPS)
    if seed == SEEDS[0]:
        saved = np.load(f'X_adv_pgd_eps01_{PIPELINE_VERSION}.npy')
        CHECK['adv_set_max_abs_diff_vs_saved'] = float(np.abs(saved - Xa).max()); print(f'seed {seed}: regenerated adversarial set vs saved .npy, max |diff| = {CHECK["adv_set_max_abs_diff_vs_saved"]:.2e}')
    m = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); m.fit(X_train, y_train, verbose=False)
    pc, pa = m.predict(X_test), m.predict(Xa)
    CM_CLEAN[seed] = confusion_matrix(y_test, pc, labels=range(6)); CM_ADV[seed] = confusion_matrix(y_test, pa, labels=range(6))
    print(f'seed {seed:<5} clean F1 {f1_score(y_test, pc, average="weighted"):.4f} | PGD F1 {f1_score(y_test, pa, average="weighted"):.4f} | {time.time() - t0:.0f}s', flush=True)

## 2. Row-normalised confusion matrices (mean over 5 seeds)

Rows = true class, columns = predicted class; each row sums to 1 (that is per-class recall on the diagonal).

In [ ]:
rn = lambda cm: cm / cm.sum(1, keepdims=True)
R_clean = np.mean([rn(CM_CLEAN[s]) for s in SEEDS], 0); R_adv = np.mean([rn(CM_ADV[s]) for s in SEEDS], 0)
R_adv_sd = np.std([rn(CM_ADV[s]) for s in SEEDS], 0, ddof=1)
df = lambda a: pd.DataFrame(a, index=[f'true {c}' for c in class_names], columns=[f'-> {c}' for c in class_names])
pd.set_option('display.width', 200)
print('CLEAN (mean of 5 seeds)'); print(df(R_clean).round(3).to_string())
print('\nPGD eps=0.1 (mean of 5 seeds)'); print(df(R_adv).round(3).to_string())
print('\nChange (PGD - clean)'); print(df(R_adv - R_clean).round(3).to_string())
print('\nSD of PGD matrix across seeds'); print(df(R_adv_sd).round(3).to_string())

print('\nWhere do the rows of each true class go under PGD? (share of the class; clean share in brackets)')
for ci, cn in enumerate(class_names):
    order = np.argsort(-R_adv[ci])
    print(f'  {cn:<13}' + '  '.join(f'{class_names[j]} {R_adv[ci, j]:.3f} ({R_clean[ci, j]:.3f})' for j in order[:3]))

print('\nSeed-42 raw counts, PGD eps=0.1'); print(pd.DataFrame(CM_ADV[42], index=class_names, columns=class_names).to_string())

## 3. Figure and save

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(19, 5.6))
for a, M, t, cm_, vm in [(ax[0], R_clean, 'Clean', 'Blues', (0, 1)), (ax[1], R_adv, 'PGD eps=0.1', 'Blues', (0, 1)), (ax[2], R_adv - R_clean, 'Change (PGD - clean)', 'RdBu_r', (-1, 1))]:
    im = a.imshow(M, cmap=cm_, vmin=vm[0], vmax=vm[1]); a.set_title(f'{t} (tuned XGBoost, 5-seed mean, row-normalised)', fontsize=10)
    a.set_xticks(range(6)); a.set_xticklabels(class_names, rotation=40, ha='right'); a.set_yticks(range(6)); a.set_yticklabels(class_names)
    a.set_xlabel('predicted'); a.set_ylabel('true')
    for i in range(6):
        for j in range(6): a.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', fontsize=8, color='white' if abs(M[i, j]) > 0.55 * vm[1] else 'black')
    plt.colorbar(im, ax=a, fraction=0.046)
plt.tight_layout(); plt.savefig(f'attack1_confusion_pgd_eps01_{PIPELINE_VERSION}.png', dpi=150); plt.show()
df(R_clean).to_csv(f'attack1_confusion_clean_{PIPELINE_VERSION}.csv'); df(R_adv).to_csv(f'attack1_confusion_pgd_eps01_{PIPELINE_VERSION}.csv')
joblib.dump({'class_names': class_names, 'seeds': SEEDS, 'eps': EPS, 'counts_clean': CM_CLEAN, 'counts_pgd': CM_ADV, 'row_norm_clean_mean': R_clean, 'row_norm_pgd_mean': R_adv,
             'row_norm_pgd_sd': R_adv_sd, 'checks': CHECK}, f'attack1_confusion_{PIPELINE_VERSION}.pkl')
for f in [f'attack1_confusion_{PIPELINE_VERSION}.pkl', f'attack1_confusion_pgd_eps01_{PIPELINE_VERSION}.png', f'attack1_confusion_clean_{PIPELINE_VERSION}.csv', f'attack1_confusion_pgd_eps01_{PIPELINE_VERSION}.csv']:
    print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)